# Week 2: Frame Your Lane as an ML Task

**Assignment:** ML-03 · **Track:** Machine Learning · **Phase:** Foundations

This notebook develops the Week 1 provisional lane into a concrete ML task. It uses only the anonymized starter dataset and describes a decision-support workflow for human review.

## 1. My lane as an ML task (type)

My lane is **Refresh / Content Opportunity Scoring**. Its ML task type is **ranking**, supported by a starter-data **classification proxy**. The practical output is a ranked queue: pages nearer the top are the pages a content team should review first.

A score is useful only because it creates an order under limited capacity. In a later model, the classification probability can be one input to that order, alongside an explicit and inspectable baseline score. The output is not an automatic instruction to edit a page. It is evidence for a content strategist to review.

## 2. Target or proxy

For this starter exercise, the target proxy is `is_declining_proxy`: a page is positive when `trend_direction == "down"`. It gives the model a concrete example of a page with an observed current-window decline signal.

This is a teaching proxy, not a future prediction target. It is derived from the same comparison window as `trend_pct`, so neither `trend_direction` nor `trend_pct` can be used as a feature. Doing so would leak the answer. For the capstone, I would define a time-separated target, for example: features from a prior window → meaningful decline in a later window.

## 3. Success metric

The primary metric is **Precision@20**: among the 20 pages placed at the top of the review queue, what fraction are positive according to the chosen target or proxy? This fits a team that can inspect about 20 pages in one review cycle.

A transparent baseline rule, such as prioritizing stale pages with meaningful visibility, is the minimum comparison. A learned ranking earns its extra complexity only if it improves Precision@20 on an appropriate holdout while still producing reason codes that a reviewer can inspect. Missing a worthwhile page and sending a reviewer to an unhelpful page are both costs, so I will also consider recall and the capacity tradeoff later.

The action supported by the output is human review. A strategist can choose to refresh, expand, protect, investigate a CTR issue, monitor, or leave a page unchanged after inspecting its evidence.

## 4. The unit of analysis, as a real dataframe

One row is one anonymized content item, or page, observed over a trailing 90-day window. `content_id` identifies the page and `client_id` identifies the pseudonymized client. They are shown for grouping and traceability only, not as model features.

The candidate signals shown below are visibility, freshness, position, click-through rate, and engagement. `trend_direction` and `trend_pct` are deliberately absent from this feature view because they encode the current target proxy.

In [1]:
from pathlib import Path

import pandas as pd

candidate_paths = [root / "data/raw/content_refresh_anonymized.csv" for root in [Path.cwd(), *Path.cwd().parents]]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv.")

df = pd.read_csv(data_path)
df["is_declining_proxy"] = df["trend_direction"].eq("down")

print(f"Loaded {len(df):,} rows. One row represents one anonymized content item.")
print("Target sketch: is_declining_proxy = (trend_direction == 'down').")

Loaded 30,000 rows. One row represents one anonymized content item.
Target sketch: is_declining_proxy = (trend_direction == 'down').


In [2]:
unit_columns = [
    "content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "engagement_rate", "is_declining_proxy"
]
unit_of_analysis = df.loc[:, unit_columns].head(8).copy()
display(unit_of_analysis)

print(f"Rows: {len(df):,}; unique content IDs: {df['content_id'].nunique():,}; unique clients: {df['client_id'].nunique():,}.")
print(f"Current decline proxy rate: {df['is_declining_proxy'].mean():.1%}.")

,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,engagement_rate,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,5.88,True
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,0.00,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,0.00,True
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,1.28,False
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,0.00,True
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,0.00,True
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,0.00,True
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,3.57,False


Rows: 30,000; unique content IDs: 30,000; unique clients: 32.
Current decline proxy rate: 54.2%.


## 5. Why ML beats a fixed rule here

A fixed rule is valuable because it is transparent. For example, a rule can surface pages that are stale and have enough impressions to matter. It is also limited because it applies the same hand-written weighting to every page.

A learned ranking can combine several observable signals, such as visibility, freshness, position, CTR, and engagement, and learn patterns that a single rule may miss. That does not make ML automatically better. It must beat the fixed-rule baseline on held-out Precision@20, avoid leakage, and keep the recommendation explainable with reason codes.

This is therefore not just a model-training exercise. The real problem is to help a person decide where to spend limited content-review time. ML is useful only if it improves that decision while preserving human judgment and clear limits on what the data can claim.

## 6. Self-check

- [x] I named the lane as a ranking task supported by a classification proxy.
- [x] I defined the starter target proxy and stated its leakage limitation.
- [x] I selected Precision@20 because it matches a real review capacity.
- [x] I loaded the starter data and showed a real page-level dataframe.
- [x] I connected the ranked output to a human content action.
- [x] I explained why ML must improve on a transparent rule rather than replace judgment.
- [x] I used careful language and did not claim a refresh causes recovery or prove a Google ranking factor.